# Get all data paths for counting/averaging later on

In [1]:
import os
import re
from typing import List, Dict, Set, Tuple
from collections import defaultdict
import multiprocessing as mp
from functools import partial
from tqdm import tqdm

def find_user_directories(root_path: str) -> List[str]:
    """
    Find all valid user directories in the root path.
    Valid directories are named with a letter followed by up to two digits.
    """
    user_dir_pattern = re.compile(r'^[A-Za-z]\d{0,2}$')
    return [d for d in os.listdir(root_path) if os.path.isdir(os.path.join(root_path, d)) and user_dir_pattern.match(d)]

def find_route_directories(user_path: str) -> List[str]:
    """
    Find all valid route directories in a user directory.
    Valid directories are named with the user prefix followed by two numbers separated by underscores.
    Also catches incorrectly named directories with just one number.
    """
    user_prefix = os.path.basename(user_path)
    route_dir_pattern = re.compile(f'^{user_prefix}_(\d+)(_\d+)?$')
    return [d for d in os.listdir(user_path) if os.path.isdir(os.path.join(user_path, d)) and route_dir_pattern.match(d)]

def analyze_dataset_structure(root_path: str) -> Dict[str, List[str]]:
    """
    Analyze the dataset structure, finding all user directories and their corresponding route directories.
    """
    dataset_structure = {}
    
    user_dirs = find_user_directories(root_path)
    for user_dir in user_dirs:
        user_path = os.path.join(root_path, user_dir)
        route_dirs = find_route_directories(user_path)
        dataset_structure[user_dir] = route_dirs
    
    return dataset_structure

def process_file(file_name: str, directory: str, ignore_list: List[str], pattern: re.Pattern) -> Tuple[str, str, str]:
    full_path = os.path.join(directory, file_name)
    if not os.path.isfile(full_path) or (ignore_list and any(ignore_text in file_name for ignore_text in ignore_list)):
        return None
    name, ext = os.path.splitext(file_name)
    modified_name = pattern.sub(r'\1\2', name)
    return (modified_name, ext, full_path)

def analyze_files(directory: str, ignore_list: List[str] = None, num_processes: int = None) -> Dict[str, Tuple[int, List[str], Set[str]]]:
    num_processes = mp.cpu_count() if num_processes is None else num_processes
    pattern = re.compile(r'(.*?)[0-9]{6}(.*)')
    
    file_list = os.listdir(directory)
    
    with mp.Pool(processes=num_processes) as pool:
        process_func = partial(process_file, directory=directory, ignore_list=ignore_list, pattern=pattern)
        results = list(tqdm(pool.imap(process_func, file_list), total=len(file_list), desc=f"Analyzing files in {os.path.basename(directory)}"))

    file_info = defaultdict(lambda: [0, set(), set()])
    for result in results:
        if result:
            modified_name, ext, full_path = result
            file_info[modified_name][0] += 1
            file_info[modified_name][1].add(ext)
            file_info[modified_name][2].add(full_path)

    return {k: (v[0], list(v[1]), v[2]) for k, v in file_info.items()}

def process_dataset(root_path: str, ignore_list: List[str] = None, num_processes: int = None) -> Dict[str, Dict[str, Dict[str, Tuple[int, List[str], Set[str]]]]]:
    """
    Process the entire dataset, analyzing files in each route directory.
    """
    dataset_structure = analyze_dataset_structure(root_path)
    processed_data = defaultdict(lambda: defaultdict(dict))

    total_routes = sum(len(routes) for routes in dataset_structure.values())
    pbar = tqdm(total=total_routes, desc="Processing routes", unit="route")

    for user_dir, route_dirs in dataset_structure.items():
        for route_dir in route_dirs:
            route_path = os.path.join(root_path, user_dir, route_dir)
            processed_data[user_dir][route_dir] = analyze_files(route_path, ignore_list, num_processes)
            pbar.update(1)
            pbar.set_postfix_str(f"Current: {user_dir}/{route_dir}")

    pbar.close()
    return processed_data

In [2]:
root_path = "/data-net/ted/"
ignore_list = ['warped', 'overlay', '.tar', '.mp4', '_frame', 'AM_image', 'gaze_data', 'homography', 'EPOCX',
               'directions_data', 'input_data', 'scenario_recording']
num_processes = 2 * mp.cpu_count()
dataset_info = process_dataset(root_path, ignore_list, num_processes)

Analyzing files in M20_1_1: 100%|██████████| 1320575/1320575 [00:44<00:00, 29406.41it/s]_4_3] 
Analyzing files in M20_6_3: 0it [00:00, ?it/s]:47<2:12:28, 62.10s/route, Current: M20/M20_1_1]
Analyzing files in M7_6_2: 100%|██████████| 911013/911013 [00:46<00:00, 19582.13it/s]/M7_1_1] 
Analyzing files in H22_10_1: 0it [00:00, ?it/s]52<2:44:56, 113.75s/route, Current: M7/M7_6_2]
Analyzing files in H22_63: 100%|██████████| 297378/297378 [00:14<00:00, 20341.09it/s]/H22_43] 
Analyzing files in H22_4_3: 0it [00:00, ?it/s]:22<1:27:37, 64.12s/route, Current: H22/H22_63]
Analyzing files in M6_1_1: 100%|██████████| 883215/883215 [00:49<00:00, 17878.70it/s]10/M10_1_1]
Analyzing files in H20_4_1: 0it [00:00, ?it/s]:52:47<17:01, 127.65s/route, Current: M6/M6_1_1]  
Processing routes: 100%|██████████| 138/138 [3:04:48<00:00, 80.35s/route, Current: H3/H3_4_2] 


In [3]:
import json
import h5py
from typing import Dict
from collections import defaultdict

def save_dataset_info_hdf5(dataset_info: Dict, filepath: str):
    """
    Save the dataset_info dictionary using HDF5.
    """
    with h5py.File(filepath, 'w') as f:
        for user, user_data in dataset_info.items():
            user_group = f.create_group(user)
            for route, route_data in user_data.items():
                route_group = user_group.create_group(route)
                for file_type, (count, extensions, paths) in route_data.items():
                    file_group = route_group.create_group(file_type)
                    file_group.attrs['count'] = count
                    file_group.create_dataset('extensions', data=json.dumps(extensions))
                    file_group.create_dataset('paths', data=json.dumps(list(paths)))
    print(f"Dataset info saved to {filepath}")

def load_dataset_info_hdf5(filepath: str) -> Dict:
    """
    Load the dataset_info dictionary using HDF5.
    """
    dataset_info = defaultdict(lambda: defaultdict(dict))
    with h5py.File(filepath, 'r') as f:
        for user in f.keys():
            for route in f[user].keys():
                for file_type in f[user][route].keys():
                    count = f[user][route][file_type].attrs['count']
                    extensions = json.loads(f[user][route][file_type]['extensions'][()])
                    paths = set(json.loads(f[user][route][file_type]['paths'][()]))
                    dataset_info[user][route][file_type] = (count, extensions, paths)
    print(f"Dataset info loaded from {filepath}")
    return dataset_info

In [4]:
# Save the dataset info
save_dataset_info_hdf5(dataset_info, "paths_all_data_27August2024.h5")

Dataset info saved to paths_all_data_27August2024.h5


In [5]:
# Later, to load the dataset info
# loaded_dataset_info = load_dataset_info_hdf5("paths_all_data.h5")

Dataset info loaded from paths_all_data.h5


In [5]:
dataset_info['H2']['H2_1_1']['rgb_left']

(18290,
 ['.jpg'],
 {'/data-net/ted/H2/H2_1_1/rgb_left016123.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left009130.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left011241.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left016393.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left008493.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left000723.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left008651.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left005247.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left000699.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left004444.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left013939.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left002228.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left004171.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left008044.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left017168.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left000011.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left011890.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left006710.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left010365.jpg',
  '/data-net/ted/H2/H2_1_1/rgb_left012647.jpg',
  '/data-net/ted/H2/H

In [6]:
from typing import Dict

# ... [Previous functions remain unchanged] ...

def print_dataset_info(dataset_info: Dict):
    """
    Print the dataset info in the specified format:
    User:
        Route:
            sensor, extension, number of files
    """
    for user, user_data in dataset_info.items():
        print(f"User: {user}")
        for route, route_data in user_data.items():
            print(f"    Route: {route}")
            for sensor, (count, extensions, _) in route_data.items():
                for ext in extensions:
                    print(f"        {sensor}, {ext}, {count}")
        print()  # Add an empty line between users for better readability

In [7]:
print_dataset_info(dataset_info)

User: H2
    Route: H2_6_2
        rgb_WetNoon_central, .jpg, 6625
        attention_mirror_left, .npz, 6975
        rgb_left, .jpg, 6622
        semantic_segmentation_right, .png, 6622
        rgb_MidRainSunset_mirror_left, .jpg, 6623
        depth_mirror_right, .png, 6622
        semantic_segmentation_mirror_right, .png, 6622
        cmd_fix_can_bus, .json, 6623
        rgb_CloudyNoon_central, .jpg, 6623
        optical_flow_left, .png, 13244
        optical_flow_left, .npy, 13244
        depth_right, .png, 6622
        attention_mirror_right, .npz, 6975
        rgb_WetNoon_left, .jpg, 6624
        rgb_central, .jpg, 6623
        rgb_mirror_left, .jpg, 6622
        rgb_ClearSunset_central, .jpg, 6625
        optical_flow_mirror_right, .png, 13244
        optical_flow_mirror_right, .npy, 13244
        optical_flow_mirror_left, .png, 13244
        optical_flow_mirror_left, .npy, 13244
        rgb_HardRainNoon_mirror_right, .jpg, 6624
        attention_right, .npz, 6975
        rgb_WetN

In [8]:
import pandas as pd

def dataset_info_to_dataframe(dataset_info: Dict) -> pd.DataFrame:
    """
    Convert the dataset_info dictionary to a pandas DataFrame.
    """
    data = []
    for user, user_data in dataset_info.items():
        for route, route_data in user_data.items():
            for sensor, (count, extensions, _) in route_data.items():
                for ext in extensions:
                    data.append({
                        'User': user,
                        'Route': route,
                        'Sensor': sensor,
                        'Extension': ext,
                        'File Count': count
                    })
    
    df = pd.DataFrame(data)
    return df


In [9]:
df = dataset_info_to_dataframe(dataset_info)

In [10]:
sensor_totals = df.groupby('Sensor')['File Count'].sum().sort_values(ascending=False)

In [11]:
pd.set_option('display.max_rows', None)



In [12]:
print(sensor_totals)

Sensor
optical_flow_left                     5149516
optical_flow_central                  5149424
optical_flow_mirror_right             5149120
optical_flow_mirror_left              5148994
optical_flow_right                    5148786
attention_mirror_right                1336482
attention_mirror_left                 1336457
attention_right                       1336365
attention_central                     1336363
attention_left                        1336356
cmd_fix_can_bus                       1314599
rgb_mirror_left                       1287561
depth_central                         1287560
semantic_segmentation_central         1287462
semantic_segmentation_mirror_right    1287458
rgb_left                              1287457
depth_mirror_right                    1287410
rgb_central                           1287375
semantic_segmentation_left            1287372
depth_mirror_left                     1287333
rgb_mirror_right                      1287330
semantic_segmentation_mirro

In [13]:
def save_dataframe_to_csv(df: pd.DataFrame, filepath: str):
    """
    Save the DataFrame to a CSV file.
    """
    df.to_csv(filepath, index=False)
    print(f"DataFrame saved to {filepath}")

def load_dataframe_from_csv(filepath: str) -> pd.DataFrame:
    """
    Load the DataFrame from a CSV file.
    """
    df = pd.read_csv(filepath)
    print(f"DataFrame loaded from {filepath}")
    return df

In [14]:
df.head()

,User,Route,Sensor,Extension,File Count
0,H2,H2_6_2,rgb_WetNoon_central,.jpg,6625
1,H2,H2_6_2,attention_mirror_left,.npz,6975
2,H2,H2_6_2,rgb_left,.jpg,6622
3,H2,H2_6_2,semantic_segmentation_right,.png,6622
4,H2,H2_6_2,rgb_MidRainSunset_mirror_left,.jpg,6623


In [15]:
save_dataframe_to_csv(df, 'all_data_count_27August2024.csv')

DataFrame saved to all_data_count_27August2024.csv


In [16]:
def custom_sort_key(value):
    # Split the value into alphabetic and numeric parts
    match = re.match(r'([A-Z])(\d+)', value)
    if match:
        alpha, num = match.groups()
        return (int(num), alpha)  # Sort first by number, then by letter
    return (float('inf'), value)  # Default sort key if the format doesn't match

def parse_route(route):
    parts = route.split('_')
    if len(parts) == 3:
        return int(parts[1]), int(parts[2])
    elif len(parts) == 2:
        return int(parts[1]), 0
    else:
        return -1, -1  # Default values for invalid formats

def custom_sort(df):
    # Create sorting keys
    df['sort_key'] = df['User'].apply(custom_sort_key)
    df['route_key1'], df['route_key2'] = zip(*df['Route'].apply(parse_route))
    
    # Sort the DataFrame
    df_sorted = df.sort_values(['sort_key', 'route_key1', 'route_key2'])
    
    # Remove helper columns
    df_sorted = df_sorted.drop(['sort_key', 'route_key1', 'route_key2'], axis=1)
    
    # Reset the index
    df_sorted = df_sorted.reset_index(drop=True)
    
    return df_sorted

def dataset_info_to_reshaped_dataframe(dataset_info: Dict) -> pd.DataFrame:
    """
    Convert the dataset_info dictionary to a reshaped pandas DataFrame.
    Columns are sensor name plus extension, rows are user and routes, cells are file counts.
    The DataFrame is sorted using a custom sorting method for users and routes.
    """
    data = []
    sensor_extensions = set()

    # First pass: collect all unique sensor-extension combinations and basic data
    for user, user_data in dataset_info.items():
        for route, route_data in user_data.items():
            row_data = {'User': user, 'Route': route}
            for sensor, (count, extensions, _) in route_data.items():
                for ext in extensions:
                    sensor_ext = f"{sensor}{ext}"
                    sensor_extensions.add(sensor_ext)
                    row_data[sensor_ext] = count
            data.append(row_data)

    # Create DataFrame
    df = pd.DataFrame(data)

    # Fill NaN values with 0 (for routes that don't have certain sensor-extension combinations)
    for sensor_ext in sensor_extensions:
        if sensor_ext not in df.columns:
            df[sensor_ext] = 0
        else:
            df[sensor_ext] = df[sensor_ext].fillna(0)

    # Sort columns alphabetically
    sensor_columns = sorted(sensor_extensions)
    df = df[['User', 'Route'] + sensor_columns]

    # Apply custom sorting
    df = custom_sort(df)

    return df

In [17]:
df_reshaped = dataset_info_to_reshaped_dataframe(dataset_info)

In [18]:
df_reshaped

,User,Route,attention_central.npz,attention_left.npz,attention_mirror_left.npz,attention_mirror_right.npz,attention_right.npz,cmd_fix_can_bus.json,depth_central.png,depth_left.png,...,rgb_central.jpg,rgb_left.jpg,rgb_mirror_left.jpg,rgb_mirror_right.jpg,rgb_right.jpg,semantic_segmentation_central.png,semantic_segmentation_left.png,semantic_segmentation_mirror_left.png,semantic_segmentation_mirror_right.png,semantic_segmentation_right.png
0,H2,H2_1_1,18951.0,18951.0,18951.0,18951.0,18951.0,18347.0,18291.0,18290.0,...,18291.0,18290.0,18290.0,18290.0,18290.0,18291.0,18290.0,18290.0,18290.0,18290.0
1,H2,H2_4_3,5799.0,5799.0,5799.0,5799.0,5799.0,5353.0,5353.0,5352.0,...,5353.0,5352.0,5352.0,5352.0,5352.0,5353.0,5352.0,5352.0,5352.0,5352.0
2,H2,H2_6_2,6975.0,6975.0,6975.0,6975.0,6975.0,6623.0,6623.0,6622.0,...,6623.0,6622.0,6622.0,6622.0,6622.0,6623.0,6622.0,6622.0,6622.0,6622.0
3,M2,M2_1_1,3783.0,3778.0,3834.0,3856.0,3780.0,15508.0,3815.0,3694.0,...,3740.0,3820.0,3895.0,3760.0,3711.0,3771.0,3778.0,3718.0,3818.0,3685.0
4,M2,M2_4_3,3641.0,3641.0,3641.0,3641.0,3641.0,3434.0,3434.0,3433.0,...,3434.0,3433.0,3433.0,3433.0,3433.0,3434.0,3433.0,3433.0,3433.0,3433.0
5,M2,M2_6_2,5101.0,5101.0,5101.0,5101.0,5101.0,4878.0,4878.0,4877.0,...,4878.0,4877.0,4877.0,4877.0,4877.0,4878.0,4877.0,4877.0,4877.0,4877.0
6,H3,H3_1_1,18606.0,18606.0,18606.0,18606.0,18606.0,18218.0,18161.0,18160.0,...,18161.0,18160.0,18160.0,18160.0,18160.0,18161.0,18160.0,18160.0,18160.0,18160.0
7,H3,H3_4_2,6973.0,6973.0,6973.0,6973.0,6973.0,6518.0,6436.0,6435.0,...,6436.0,6435.0,6435.0,6435.0,6435.0,6436.0,6435.0,6435.0,6435.0,6435.0
8,H3,H3_10_1,8443.0,8443.0,8443.0,8443.0,8443.0,7993.0,7993.0,7992.0,...,7993.0,7992.0,7992.0,7993.0,7992.0,7993.0,7992.0,7992.0,7992.0,7992.0
9,M3,M3_1_1,18051.0,18051.0,18051.0,18051.0,18051.0,17604.0,17753.0,17753.0,...,17752.0,17752.0,17752.0,17751.0,17752.0,17753.0,17753.0,17754.0,17751.0,17751.0


In [21]:
len(df_reshaped.User.unique())

44

In [58]:
# cols_to_drop = df_reshaped.columns[df_reshaped.columns.str.contains('directions_data')]
# df_reshaped.drop(cols_to_drop, axis=1, inplace=True)

In [59]:
# cols_to_drop = df_reshaped.columns[df_reshaped.columns.str.contains('input_data')]
# df_reshaped.drop(cols_to_drop, axis=1, inplace=True)

In [60]:
# cols_to_drop = df_reshaped.columns[df_reshaped.columns.str.contains('scenario_recording')]
# df_reshaped.drop(cols_to_drop, axis=1, inplace=True)

In [61]:
# cols_to_drop = df_reshaped.columns[df_reshaped.columns.str.contains('EPOCX')]
# df_reshaped.drop(cols_to_drop, axis=1, inplace=True)

In [64]:
# cols_to_drop = df_reshaped.columns[df_reshaped.columns.str.contains('gaze_data')]
# df_reshaped.drop(cols_to_drop, axis=1, inplace=True)

In [65]:
# cols_to_drop = df_reshaped.columns[df_reshaped.columns.str.contains('homography')]
# df_reshaped.drop(cols_to_drop, axis=1, inplace=True)

In [22]:
df_reshaped.to_csv('all_data_count_August27_2024_nicer_sorted.csv', index=False)